In [10]:
import dhg
# DHG requires List[List[int]] for hyperedges
edges = [(0,1),(0,2),(1,2),(3,4)]
formattedEdges = [list(e) for e in edges]

g = dhg.Graph(5,formattedEdges)
# 图的边
print("边",g.v)
# 二元组 (Tuple)，格式为 (边列表, 权重列表)。
print(g.e)
e_list,e_weight = g.e
print("边",e_list)
print("边的权重",e_weight)
print(g.e_both_side)
print(g.A)

边 [0, 1, 2, 3, 4]
([(0, 1), (0, 2), (1, 2), (3, 4)], [1.0, 1.0, 1.0, 1.0])
边 [(0, 1), (0, 2), (1, 2), (3, 4)]
边的权重 [1.0, 1.0, 1.0, 1.0]
([(0, 1), (0, 2), (1, 2), (3, 4), (1, 0), (2, 0), (2, 1), (4, 3)], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])
tensor(indices=tensor([[0, 0, 1, 1, 2, 2, 3, 4],
                       [1, 2, 0, 2, 0, 1, 4, 3]]),
       values=tensor([1., 1., 1., 1., 1., 1., 1., 1.]),
       size=(5, 5), nnz=8, layout=torch.sparse_coo)


### 方法理解
#### g.e
当你打印 g.e 时，它返回了一个二元组 (Tuple)，格式为 (边列表, 权重列表)。
```python
(
    [(0, 1), (0, 2), (1, 2), (3, 4)],  # 第一部分：边的拓扑结构 (Edges)
    [1.0, 1.0, 1.0, 1.0]              # 第二部分：边的权重 (Weights)
)
```
第一部分：拓扑结构 [(0, 1), ...]
这是图中存在的边。虽然你输入时使用的是 list（为了兼容超图），但在普通图 dhg.Graph 中，它会自动将其标准化为元组 (u, v)。

(0, 1) 表示节点 0 和节点 1 之间有一条边。

第二部分：权重 [1.0, 1.0, 1.0, 1.0]
这是每条边对应的权重 (Weight)。

因为你创建图时没有指定权重，DHG 默认给每条边分配了权重 1.0。

在图神经网络（GCN 等）的计算中，这些权重决定了消息传递时邻居节点贡献的“强度”。

> 那么如何自定义权重呢？
```python
# 假设 0-1 边权重为 5.0，其他为 1.0
weights = [5.0, 1.0, 1.0, 1.0]
g_weighted = dhg.Graph(5, formattedEdges, e_weight=weights)

print(g_weighted.e)
# 输出: ([(0, 1), (0, 2), (1, 2), (3, 4)], [5.0, 1.0, 1.0, 1.0])
```

In [ ]:
# 邻接矩阵
print(g.A.to_dense())

tensor([[0., 1., 1., 0., 0.],
        [1., 0., 1., 0., 0.],
        [1., 1., 0., 0., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0.]])


1. 矩阵坐标与边的对应关系矩阵的第 $i$ 行、第 $j$ 列的值表示节点 $i$ 和节点 $j$ 之间是否有边：第一行 (节点 0)：[0., 1., 1., 0., 0.]A[0, 1] = 1：节点 0 连接节点 1。A[0, 2] = 1：节点 0 连接节点 2。A[0, 0] = 0：目前没有自环（节点 0 没连自己）。对称性：注意 A[0, 1] 和 A[1, 0] 都是 1.。这就是为什么 g.e_both_side 有 8 条边的原因——矩阵里总共有 8 个 1.。孤立区域：最后两行 [0, 0, 0, 0, 1] 和 [0, 0, 0, 1, 0]。这代表节点 3 和 4 互相连接，但它们与节点 0, 1, 2 没有任何往来。这是一个非连通图。

2. 在 DHG 内部，g.A 默认是以稀疏矩阵 (Sparse Tensor) 存储的（只记录非零值的位置），因为在真实场景中（比如有 10 万个节点），绝大多数位置都是 0，存成稠密矩阵会直接撑爆显存。

g.A：适合大规模计算，节省空间。

g.A.to_dense()：适合我们人类观察、调试，或者在小型数据集上进行某些特定的矩阵运算。

In [15]:

# 定义带有重边 (0, 2) 的边列表
raw_edges = [(0, 1), (0, 2), (0, 2), (3, 4)]

# 方案 A: 使用平均合并 (Mean)
g_mean = dhg.Graph(5, raw_edges, merge_op="mean") # type: ignore
print("--- Merge Op: MEAN ---")
print(f"Edges & Weights: {g_mean.e}")
print(f"Vertex Degrees:  {g_mean.deg_v}")

# 方案 B: 使用累加合并 (Sum)
g_sum = dhg.Graph(5, raw_edges, merge_op="sum") # type: ignore
print("\n--- Merge Op: SUM ---")
print(f"Edges & Weights: {g_sum.e}")
print(f"Vertex Degrees:  {g_sum.deg_v}")



--- Merge Op: MEAN ---
Edges & Weights: ([(0, 1), (0, 2), (3, 4)], [1.0, 1.0, 1.0])
Vertex Degrees:  [2.0, 1.0, 1.0, 1.0, 1.0]

--- Merge Op: SUM ---
Edges & Weights: ([(0, 1), (0, 2), (3, 4)], [1.0, 2.0, 1.0])
Vertex Degrees:  [3.0, 1.0, 2.0, 1.0, 1.0]


太棒了，这套实验数据是理解图卷积（GCN）数学基础的“金钥匙”。为了方便你保存到 GitHub 仓库作为 **环境验证脚本** 或 **学习笔记**，我为你整理了这份精简、专业的 Markdown 文档。

---

# 📊 DHG 核心逻辑实验笔记：加权图与合并操作



##  结果深度解析

### 核心对比表

| 特性 | `merge_op="mean"` | `merge_op="sum"` | 深度学习意义 |
| --- | --- | --- | --- |
| **重边 (0, 2) 处理** | 权重取均值 (1.0) | 权重累加 (2.0) | 体现连接强度的差异 |
| **节点 0 的度数** | **2.0** (1.0 + 1.0) | **3.0** (1.0 + 2.0) | 影响 GCN 的归一化系数 |
| **邻接矩阵 A[0, 2]** | `1.0` | `2.0` | 改变消息传递时的贡献量 |

### 关键点解释

#### 1. **节点度数 (Vertex Degree)**：
在加权图中，节点的度 $D_i$ 定义为与其相连的所有边的权重之和：

$$d(v_i) = \sum_{j} A_{ij}$$



这就是为什么在 `sum` 模式下，节点 0 的度数变成了 `3.0`，而节点 2 变成了 `2.0`。
#### 2. **消息传递 (Message Passing)**：
在图卷积中，节点 $j$ 传递给节点 $i$ 的信息量通常与 $A_{ij}$ 成正比。使用 `sum` 合并意味着模型会认为节点 0 和 2 之间的“共同话题”更多，信息流动更强。

## 3. 动态边操作 (Incremental Updates)

使用 `add_edges` 可以动态改变图的拓扑和权重。`merge_op` 会决定新旧权重的融合方式。

### 实验记录
1. **初始状态**: 边 (0,1) 权重 1.0
2. **Mean 合并**: 再次添加 (1,0), 权重保持为 `(1.0+1.0)/2 = 1.0`
3. **Sum 合并**: 再次添加 (1,0), 权重累加为 `1.0+1.0 = 2.0`

### 适用场景
- **Mean**: 适用于多源数据融合，防止单一连接因重复出现而导致权重过大（归一化）。
- **Sum**: 适用于计数场景，重复出现的连接代表更强的先验置信度。


In [16]:
# 实验数据：定义 5 个顶点的邻接列表
# [源点, 邻居1, 邻居2, ...]
adj_list = [
    [0, 1, 2],       # 0 -> 1, 0 -> 2
    [1, 3],          # 1 -> 3
    [4, 3, 0, 2, 1]  # 4 -> 3, 4 -> 0, 4 -> 2, 4 -> 1
]

# 构建图
g = dhg.Graph.from_adj_list(5, adj_list)

print("--- 边列表 (Edges) ---")
print(g.e)

print("\n--- 邻接矩阵 (Adjacency Matrix) ---")
print(g.A.to_dense())

--- 边列表 (Edges) ---
([(0, 1), (0, 2), (1, 3), (3, 4), (0, 4), (2, 4), (1, 4)], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])

--- 邻接矩阵 (Adjacency Matrix) ---
tensor([[0., 1., 1., 0., 1.],
        [1., 0., 0., 1., 1.],
        [1., 0., 0., 0., 1.],
        [0., 1., 0., 0., 1.],
        [1., 1., 1., 1., 0.]])


这是一份关于使用 `dhg.Graph.from_adj_list()` 进行图构建的实验笔记。通过邻接列表（Adjacency List）构建图是处理稀疏连接数据时最常用的方式，特别是在社交网络、引用网络等领域。

---

# 📝 DHG 实验笔记：基于邻接列表构建图



## 2. 逻辑拆解与可视化

### 邻接列表的本质

邻接列表不仅定义了连接关系，还隐含了图的稀疏性。对于 `[4, 3, 0, 2, 1]` 这一条记录，它实际上创建了 4 条边：`(4, 3), (4, 0), (4, 2), (4, 1)`。

### 为什么生成的边与输入列表看起来不完全一致？

观察你的输出：

* `g.e` 结果包含了 `(0, 4), (2, 4), (1, 4)` 等边。
* 这是因为 `dhg.Graph` 构建的是**无向图**。即使你的输入列表看起来有方向性（比如 `4` 在首位），`DHG` 会自动将其对称化。如果你在列表里写了 `4 -> 0`，构建出来的图会自动包含 `0 -> 4`，这保证了图的无向对称属性。

---

## 3. 核心概念对比表

| 特性 | 描述 |
| --- | --- |
| **源点 (Source)** | 列表中索引为 0 的元素。 |
| **汇点 (Target)** | 列表中索引 1 及之后的所有元素。 |
| **图类型** | `dhg.Graph` 默认构造为无向图。 |
| **对称性** | 任何输入 `(u, v)` 都会被自动补齐为 `(v, u)`。 |

---
